In [15]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)

# --------------------------
# CONFIGURATION
# --------------------------
num_orders = 8000
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 6, 30)

hubs = {
    1: {"name": "Dubai", "country": "UAE", "capacity": 120},
    2: {"name": "Mumbai", "country": "India", "capacity": 150},
    3: {"name": "Hong Kong", "country": "China", "capacity": 100},
    4: {"name": "New York", "country": "USA", "capacity": 90}
}

product_categories = ["Loose Diamond", "Jewelry", "Custom"]
priority_types = ["Standard", "Express"]
shipping_partners = ["DHL", "FedEx", "UPS"]
shipping_zones = ["Zone 1", "Zone 2", "Zone 3"]

sla_hours = {
    "Intake": 6,
    "QC": 12,
    "Packing": 6,
    "Shipping": 12,
    "Delivery": 72
}

# --------------------------
# ORDERS DATA
# --------------------------
orders = []

for i in range(1, num_orders + 1):
    order_date = start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
    hub_id = random.choice(list(hubs.keys()))
    priority = np.random.choice(priority_types, p=[0.75, 0.25])
    order_value = np.random.normal(5000, 1500)
    
    promised_delivery = order_date + timedelta(days=5 if priority == "Express" else 7)

    orders.append([
        i,
        order_date,
        random.choice(["UAE", "India", "USA", "China", "UK", "Germany"]),
        random.choice(["UAE", "India", "USA", "China", "UK", "Germany"]),
        hub_id,
        random.choice(product_categories),
        max(500, round(order_value,2)),
        priority,
        promised_delivery
    ])

orders_df = pd.DataFrame(orders, columns=[
    "order_id", "order_date", "customer_country",
    "destination_country", "hub_id",
    "product_category", "order_value",
    "priority_flag", "promised_delivery_date"
])

orders_df.to_csv(r"D:\Personal\Data Science\Data Science\Revision\End to End Project\Nivoda\orders.csv", index=False)

# --------------------------
# FULFILMENT EVENTS
# --------------------------
events = []

for _, row in orders_df.iterrows():
    current_time = row["order_date"]
    hub_id = row["hub_id"]

    for stage in ["Intake", "QC", "Packing", "Shipping", "Delivery"]:
        
        # Inject delay patterns
        base_hours = sla_hours[stage]
        
        # Mumbai QC bottleneck
        if hub_id == 2 and stage == "QC":
            duration = np.random.normal(base_hours + 5, 4)
        else:
            duration = np.random.normal(base_hours, 3)

        # Express faster intake but pressure risk
        if row["priority_flag"] == "Express" and stage in ["Intake", "Packing"]:
            duration *= 0.7

        # Random SLA breach injection
        if random.random() < 0.12:
            duration *= 1.8

        duration = max(1, duration)
        end_time = current_time + timedelta(hours=duration)

        # Inject missing stage_end_time (messy data)
        if random.random() < 0.02:
            end_time = None

        status = "Completed"

        # QC rework injection
        if stage == "QC" and random.random() < 0.08:
            status = "Rework"

        events.append([
            row["order_id"],
            stage,
            current_time,
            end_time,
            status,
            hub_id
        ])

        current_time = end_time if end_time else current_time + timedelta(hours=duration)

events_df = pd.DataFrame(events, columns=[
    "order_id", "stage_name", "stage_start_time",
    "stage_end_time", "status", "hub_id"
])

events_df.to_csv(r"D:\Personal\Data Science\Data Science\Revision\End to End Project\Nivoda\fulfilment_events.csv", index=False)

# --------------------------
# SHIPPING COSTS
# --------------------------
shipping_data = []

for order_id in orders_df["order_id"]:
    partner = random.choice(shipping_partners)
    zone = random.choice(shipping_zones)
    weight = random.choice(["Light", "Medium", "Heavy"])
    
    cost = np.random.normal(80, 20)
    cost = max(20, round(cost,2))

    shipping_data.append([order_id, partner, zone, weight, cost])

shipping_df = pd.DataFrame(shipping_data, columns=[
    "order_id", "shipping_partner",
    "shipping_zone", "weight_bucket", "shipping_cost"
])

shipping_df.to_csv(r"D:\Personal\Data Science\Data Science\Revision\End to End Project\Nivoda\shipping_costs.csv", index=False)

# --------------------------
# RETURNS
# --------------------------
returns = []

for _, row in orders_df.iterrows():
    if random.random() < 0.06:
        returns.append([
            row["order_id"],
            random.choice(["Damaged", "Late Delivery", "Quality Issue"]),
            row["order_date"] + timedelta(days=random.randint(5, 15)),
            round(row["order_value"] * np.random.uniform(0.5, 1),2)
        ])

returns_df = pd.DataFrame(returns, columns=[
    "order_id", "return_reason",
    "return_request_date", "refund_amount"
])

returns_df.to_csv(r"D:\Personal\Data Science\Data Science\Revision\End to End Project\Nivoda\returns.csv", index=False)

# --------------------------
# HUBS
# --------------------------
hub_data = []

for hub_id, data in hubs.items():
    hub_data.append([
        hub_id,
        data["name"],
        data["country"],
        data["capacity"],
        round(np.random.uniform(15, 25),2)
    ])

hubs_df = pd.DataFrame(hub_data, columns=[
    "hub_id", "hub_name",
    "hub_country", "capacity_per_day",
    "cost_per_order"
])

hubs_df.to_csv(r"D:\Personal\Data Science\Data Science\Revision\End to End Project\Nivoda\hubs.csv", index=False)

print("Data generation complete.")

Data generation complete.


In [17]:
# --------------------------
# SLA DEFINITIONS
# --------------------------

sla_data = [
    ["Intake", 6],
    ["QC", 12],
    ["Packing", 6],
    ["Shipping", 12],
    ["Delivery", 72]
]

sla_df = pd.DataFrame(sla_data, columns=["stage_name", "sla_hours"])
sla_df.to_csv(r"D:\Personal\Data Science\Data Science\Revision\End to End Project\Nivoda\SLA_definitions.csv", index=False)

print("SLA_definitions.csv created.")

SLA_definitions.csv created.


In [4]:
sla_df.head(4)

,stage_name,sla_hours
0,Intake,6
1,QC,12
2,Packing,6
3,Shipping,12


In [7]:
pip install psycopg2-binary pandas sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
## this code is useful when the table is created in postgres and we only need to connect python with postgres and load csv into the table 

import pandas as pd
from sqlalchemy import create_engine, text

# -----------------------
# 1. Database Connection
# -----------------------


username = "postgres"
password = "sususuju3"
host = "localhost"
port = "5432"
database = "nivoda_operations"

engine = create_engine(
    f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

# -----------------------
# 2. Create Schema
# -----------------------
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS ops2;"))
    conn.commit()

# -----------------------
# 3. Load CSV Files
# -----------------------

# Orders
# orders_df = pd.read_csv("D:/data/orders.csv")

orders_df.to_sql(
    name="orders",
    con=engine,
    schema="ops2",
    if_exists="replace",   # use "append" later
    index=False
)

# Fulfilment Events
# events_df = pd.read_csv("D:/data/fulfilment_events.csv")

events_df.to_sql(
    name="fulfilment_events",
    con=engine,
    schema="ops2",
    if_exists="replace", # differnece between rep
    index=False
)

# Hubs
# hubs_df = pd.read_csv("D:/data/hubs.csv")

hubs_df.to_sql(
    name="hubs",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)

# Shipping Costs
# shipping_df = pd.read_csv("D:/data/shipping_costs.csv")

shipping_df.to_sql(
    name="shipping",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)

# Returns
# returns_df = pd.read_csv("D:/data/returns.csv")

returns_df.to_sql(
    name="returns",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)


# SLA Definitions
# sla_df = pd.read_csv("D:/data/SLA_definitions.csv")

sla_df.to_sql(
    name="sla",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)

print("✅ All tables created and data loaded successfully!")

✅ All tables created and data loaded successfully!


In [ ]:
## this code creates the schema and the table in postgres but the data type is not specified for each column so its more like a bulk load, the data type you have to change in postgres

import pandas as pd
from sqlalchemy import create_engine, text

# -----------------------
# 1. Database Connection
# -----------------------


username = "postgres"
password = "sususuju3"
host = "localhost"
port = "5432"
database = "nivoda_operations"

engine = create_engine(
    f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

# -----------------------
# 2. Create Schema
# -----------------------
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS ops2;"))
    conn.commit()

# -----------------------
# 3. Load CSV Files
# -----------------------

# Orders
# orders_df = pd.read_csv("D:/data/orders.csv")

orders_df.to_sql(
    name="orders",
    con=engine,
    schema="ops2",
    if_exists="replace",   # use "append" later
    index=False
)

# Fulfilment Events
# events_df = pd.read_csv("D:/data/fulfilment_events.csv")

events_df.to_sql(
    name="fulfilment_events",
    con=engine,
    schema="ops2",
    if_exists="replace", # differnece between rep
    index=False
)

# Hubs
# hubs_df = pd.read_csv("D:/data/hubs.csv")

hubs_df.to_sql(
    name="hubs",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)

# Shipping Costs
# shipping_df = pd.read_csv("D:/data/shipping_costs.csv")

shipping_df.to_sql(
    name="shipping",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)

# Returns
# returns_df = pd.read_csv("D:/data/returns.csv")

returns_df.to_sql(
    name="returns",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)


# SLA Definitions
# sla_df = pd.read_csv("D:/data/SLA_definitions.csv")

sla_df.to_sql(
    name="sla",
    con=engine,
    schema="ops2",
    if_exists="replace",
    index=False
)

5

In [18]:
orders_df.isna().sum()

order_id                  0
order_date                0
customer_country          0
destination_country       0
hub_id                    0
product_category          0
order_value               0
priority_flag             0
promised_delivery_date    0
dtype: int64

In [19]:
events_df.isna().sum()

order_id              0
stage_name            0
stage_start_time      0
stage_end_time      722
status                0
hub_id                0
dtype: int64

In [20]:
hubs_df.isna().sum()

hub_id              0
hub_name            0
hub_country         0
capacity_per_day    0
cost_per_order      0
dtype: int64

In [21]:
returns_df.isna().sum()

order_id               0
return_reason          0
return_request_date    0
refund_amount          0
dtype: int64

In [22]:
shipping_df.isna().sum()

order_id            0
shipping_partner    0
shipping_zone       0
weight_bucket       0
shipping_cost       0
dtype: int64

In [ ]:
## this one creates the table with the data types specified for each columns as well and then load csv into it 

import pandas as pd
from sqlalchemy import create_engine, text

# -----------------------------
# 1. DB CONNECTION
# -----------------------------
username = "postgres"
password = "sususuju3"
host = "localhost"
port = "5432"
database = "nivoda_operations"

engine = create_engine(
    f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

# -----------------------------
# 2. CREATE SCHEMA + TABLES
# -----------------------------
create_tables_sql = """
CREATE SCHEMA IF NOT EXISTS ops1;

CREATE TABLE IF NOT EXISTS ops1.orders (
    order_id BIGINT PRIMARY KEY,
    order_date DATE,
    customer_country VARCHAR(100),
    destination_country VARCHAR(100),
    hub_id INT,
    product_category VARCHAR(100),
    order_value NUMERIC(10,2),
    priority_flag VARCHAR(20),
    promised_delivery_date DATE
);

CREATE TABLE IF NOT EXISTS ops1.fulfilment_events (
    event_id BIGSERIAL PRIMARY KEY,
    order_id BIGINT,
    stage_name VARCHAR(100),
    stage_start_time TIMESTAMP,
    stage_end_time TIMESTAMP,
    status VARCHAR(50),
    hub_id INT
);

CREATE TABLE IF NOT EXISTS ops1.hubs (
hub_id INT Primary key,
hub_name Varchar(50),
hub_country Varchar(10),
capacity_per_day INT,
cost_per_order NUmeric(10,2)
);

CREATE TABLE IF NOT EXISTS ops1.shipping (
shipped_id BIGSERIAL Primary key,
order_id INT ,
shipping_partner Varchar(50),
shipping_zone Varchar(10),
weight_bucket Varchar(20),
shipping_cost NUmeric(10,2)
);

CREATE TABLE IF NOT EXISTS ops1.returns (
return_id BIGSERIAL PRIMARY KEY,
order_id INT ,
return_reason Varchar(50),
return_request_date DATE,
refund_amount Numeric(10,2)
);

CREATE TABLE IF NOT EXISTS ops1.sla (
stage_name varchar(20),
sla_hours INT
);
"""

with engine.connect() as conn:
    conn.execute(text(create_tables_sql))
    conn.commit()

print("✅ Schema and tables created successfully!")

# -----------------------------
# 3. LOAD CSV FILES
# -----------------------------

# Change paths accordingly
# orders_df = pd.read_csv("D:/data/orders.csv")
orders_df.to_sql("orders", engine, schema="ops1", if_exists="append", index=False)

# events_df = pd.read_csv("D:/data/fulfilment_events.csv")
events_df.to_sql("fulfilment_events", engine, schema="ops1", if_exists="append", index=False)

# hubs_df = pd.read_csv("D:/data/hubs.csv")
hubs_df.to_sql("hubs", engine, schema="ops1", if_exists="append", index=False)

# shipping_df = pd.read_csv("D:/data/shipping_costs.csv")
shipping_df.to_sql("shipping", engine, schema="ops1", if_exists="append", index=False)

# returns_df = pd.read_csv("D:/data/returns.csv")
returns_df.to_sql("returns", engine, schema="ops1", if_exists="append", index=False)

# sla_df = pd.read_csv("D:/data/SLA_definitions.csv")
sla_df.to_sql("sla", engine, schema="ops1", if_exists="append", index=False)

print("🚀 Data loaded successfully!")

✅ Schema and tables created successfully!
🚀 Data loaded successfully!


In [2]:
orders_n = pd.read_csv(r"D:\Personal\Data Science\Data Science\Revision\End to End Project\Nivoda\orders.csv")

orders_n.head() 

NameError: name 'pd' is not defined

In [8]:
events_df.head()

,order_id,stage_name,stage_start_time,stage_end_time,status,hub_id
0,1,Intake,2025-06-03 00:00:00.000000,2025-06-03 06:47:05.552847,Completed,4
1,1,QC,2025-06-03 06:47:05.552847,2025-06-03 19:31:23.624076,Completed,4
2,1,Packing,2025-06-03 19:31:23.624076,2025-06-04 01:21:37.674365,Completed,4
3,1,Shipping,2025-06-04 01:21:37.674365,2025-06-04 09:28:11.018142,Completed,4
4,1,Delivery,2025-06-04 09:28:11.018142,2025-06-07 12:04:55.847702,Completed,4


In [11]:
hubs_df.head()

,hub_id,hub_name,hub_country,capacity_per_day,cost_per_order
0,1,Dubai,UAE,120,20.59
1,2,Mumbai,India,150,15.90
2,3,Hong Kong,China,100,16.15
3,4,New York,USA,90,23.89


In [12]:
returns_df.head()

,order_id,return_reason,return_request_date,refund_amount
0,42,Late Delivery,2025-01-30,2961.86
1,99,Late Delivery,2025-02-28,5617.88
2,104,Quality Issue,2025-06-13,5491.34
3,165,Damaged,2025-02-27,3910.60
4,186,Quality Issue,2025-02-16,4014.88


In [14]:
returns_df["return_reason"].str.len().max()

np.int64(13)

In [15]:
shipping_df.head()

,order_id,shipping_partner,shipping_zone,weight_bucket,shipping_cost
0,1,FedEx,Zone 2,Light,89.16
1,2,FedEx,Zone 2,Medium,88.37
2,3,FedEx,Zone 2,Medium,91.18
3,4,DHL,Zone 1,Heavy,96.33
4,5,DHL,Zone 1,Light,82.47


In [16]:
sla_df.head()

,stage_name,sla_hours
0,Intake,6
1,QC,12
2,Packing,6
3,Shipping,12
4,Delivery,72


In [1]:
len(orders_df)

NameError: name 'orders_df' is not defined

In [5]:
events_df.isna().sum()

order_id              0
stage_name            0
stage_start_time      0
stage_end_time      765
status                0
hub_id                0
dtype: int64

In [6]:
len(events_df)

40000

In [9]:
events_df.to_csv("fulfilment_events.csv", index=False)

In [11]:
pd.read_csv("fulfilment_events.csv").isna().sum()

order_id              0
stage_name            0
stage_start_time      0
stage_end_time      765
status                0
hub_id                0
dtype: int64